# Hybrid Retrieval Pipeline v3
**BM25 bigram + Dense (ChromaDB) + RRF + Reranker + Chunk Expand**

```
Query → [BM25 bigram top-30] ─┐
                               ├─ RRF → top-10 → Reranker → dedup(1) → top-3
Query → [Dense top-30]        ┘         └→ expand: lấy full điều luật từ parquet
```

### Thay đổi so với v2
| # | Thay đổi | Lý do |
|---|---|---|
| 1 | `BM25_TOP_K` 50→30, `DENSE_TOP_K` 50→30 | Giảm latency, chất lượng top-3 không đổi |
| 2 | `RERANK_TOP_N` 20→10 | Bottleneck chính — giảm gần nửa thời gian rerank |
| 3 | `FINAL_TOP_K` 5→3, `MAX_CHUNKS_PER_DOC` 2→1 | Thay bằng chunk expand |
| 4 | **Chunk expand** legal | Sau retrieve top-3, ghép full điều luật từ parquet |
| 5 | **Forms**: bỏ pipeline nếu detect keyword | Trả thẳng form → không cần BM25/reranker |
| 6 | **Examples**: filter theo form candidate | Retrieve đúng loại form ngay từ đầu |

## 0. Cài đặt

In [1]:
# !pip3 install rank-bm25 sentence-transformers chromadb unicodedata2 tqdm pandas pyarrow

## 1. Imports & Config

In [2]:
import re
import time
import pickle
import unicodedata
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import torch
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")

Imports OK
PyTorch : 2.9.1+cu126
Device  : cuda


In [3]:
# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path(".")
CHUNK_DIR    = PROJECT_ROOT / "dataset" / "chunks"
CHROMA_DIR   = PROJECT_ROOT / "dataset" / "chromadb"
BM25_DIR     = PROJECT_ROOT / "dataset" / "bm25"
MODEL_EMBED  = PROJECT_ROOT / "models" / "embedding"
MODEL_RERANK = PROJECT_ROOT / "models" / "reranker"

EMBED_MODEL_NAME  = "intfloat/multilingual-e5-large-instruct"
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Retrieval params — tối ưu cho top-3 + expand
BM25_TOP_K    = 30   # giảm từ 50 → ít ảnh hưởng chất lượng top-3
DENSE_TOP_K   = 30   # giảm từ 50
RRF_K         = 60
RERANK_TOP_N  = 10   # giảm từ 20 → nhanh ~2x bước rerank
FINAL_TOP_K   = 3    # giảm từ 5 → đủ cho RAG sau expand
MAX_CHUNKS_PER_DOC = 1  # dedup chặt vì đã có expand

# Expand params
MAX_EXPAND_CHUNKS = 6   # giới hạn chunk tối đa khi expand 1 điều luật (~2400 từ)

INSTRUCTIONS = {
    "legal"   : "Instruct: Tim dieu khoan phap luat Viet Nam lien quan.\nQuery: ",
    "forms"   : "Instruct: Tim mau bieu hanh chinh phu hop voi nhu cau soan thao.\nQuery: ",
    "examples": "Instruct: Tim vi du van ban hanh chinh tuong tu tinh huong sau.\nQuery: ",
}

BM25_DIR.mkdir(parents=True, exist_ok=True)
MODEL_EMBED.mkdir(parents=True, exist_ok=True)
MODEL_RERANK.mkdir(parents=True, exist_ok=True)

for name, path in {
    "legal_chunks"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms_chunks"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples_chunks": CHUNK_DIR / "examples_chunks.parquet",
    "chromadb"       : CHROMA_DIR,
}.items():
    print(f"  {'✅' if path.exists() else '❌'} {name}: {path}")

  ✅ legal_chunks: dataset/chunks/legal_chunks.parquet
  ✅ forms_chunks: dataset/chunks/forms_chunks.parquet
  ✅ examples_chunks: dataset/chunks/examples_chunks.parquet
  ✅ chromadb: dataset/chromadb


## 2. Text Normalization & Bigram Tokenization

In [4]:
def remove_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    stripped = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    return stripped.replace("đ", "d").replace("Đ", "D")


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)


def tokenize_for_bm25(text: str, use_bigrams: bool = True) -> List[str]:
    """
    Bigram tokenization: build bigrams từ all_tokens (>=1 ký tự)
    nhưng chỉ giữ unigrams >=2 ký tự để tránh nhiễu single-char.
    → Giữ được bigram "giai_o", "o_co" dù "o" bị loại khỏi unigrams.
    """
    text = normalize_text(text)
    text = remove_diacritics(text)
    all_tokens = re.findall(r"[a-z][a-z0-9]*", text)
    unigrams   = [t for t in all_tokens if len(t) >= 2]
    if not use_bigrams:
        return unigrams
    bigrams = [f"{a}_{b}" for a, b in zip(all_tokens, all_tokens[1:])]
    return unigrams + bigrams


# Test
for t in ["Quy định về hòa giải ở cơ sở", "trách nhiệm của Bộ trưởng trong ban hành văn bản"]:
    tokens = tokenize_for_bm25(t)
    uni = [x for x in tokens if '_' not in x]
    bi  = [x for x in tokens if '_' in x]
    print(f"Input   : {t}")
    print(f"Unigrams: {uni}")
    print(f"Bigrams : {bi}\n")

Input   : Quy định về hòa giải ở cơ sở
Unigrams: ['quy', 'dinh', 've', 'hoa', 'giai', 'co', 'so']
Bigrams : ['quy_dinh', 'dinh_ve', 've_hoa', 'hoa_giai', 'giai_o', 'o_co', 'co_so']

Input   : trách nhiệm của Bộ trưởng trong ban hành văn bản
Unigrams: ['trach', 'nhiem', 'cua', 'bo', 'truong', 'trong', 'ban', 'hanh', 'van', 'ban']
Bigrams : ['trach_nhiem', 'nhiem_cua', 'cua_bo', 'bo_truong', 'truong_trong', 'trong_ban', 'ban_hanh', 'hanh_van', 'van_ban']



## 3. Build & Load BM25 Indexes

In [5]:
def build_bm25_index(
    df: pd.DataFrame,
    text_col: str,
    save_path: Path,
    name: str,
    force_rebuild: bool = False,
) -> Tuple[BM25Okapi, List[str]]:
    meta_path = save_path.with_suffix(".meta.pkl")

    if save_path.exists() and meta_path.exists() and not force_rebuild:
        print(f"[{name}] Loading from cache: {save_path}")
        t0 = time.time()
        with open(save_path, "rb") as f:
            bm25 = pickle.load(f)
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        print(f"  Loaded {len(meta['chunk_ids']):,} docs in {time.time()-t0:.1f}s")
        return bm25, meta["chunk_ids"]

    print(f"[{name}] Building BM25 bigram index ({len(df):,} docs)...")
    t0 = time.time()
    chunk_ids     = df["chunk_id"].tolist()
    corpus_tokens = [
        tokenize_for_bm25(text)
        for text in tqdm(df[text_col], desc=f"  Tokenizing {name}", leave=False)
    ]
    bm25 = BM25Okapi(corpus_tokens)

    with open(save_path, "wb") as f:
        pickle.dump(bm25, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(meta_path, "wb") as f:
        pickle.dump({"chunk_ids": chunk_ids}, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"  Done in {time.time()-t0:.1f}s | {save_path.stat().st_size/1024/1024:.1f} MB")
    return bm25, chunk_ids


print("BM25 builder defined.")

BM25 builder defined.


In [6]:
FORCE_REBUILD = False

print("Loading chunk files...")
df_legal    = pd.read_parquet(CHUNK_DIR / "legal_chunks.parquet")
df_forms    = pd.read_parquet(CHUNK_DIR / "forms_chunks.parquet")
df_examples = pd.read_parquet(CHUNK_DIR / "examples_chunks.parquet")
print(f"  legal    : {len(df_legal):,} chunks")
print(f"  forms    : {len(df_forms):,} chunks")
print(f"  examples : {len(df_examples):,} chunks")
print()

# Dùng chung cache v2 (bigram, không cần rebuild)
bm25_legal,    ids_legal    = build_bm25_index(df_legal,    "text", BM25_DIR / "bm25_legal_v2.pkl",    "legal",    FORCE_REBUILD)
bm25_forms,    ids_forms    = build_bm25_index(df_forms,    "text", BM25_DIR / "bm25_forms_v2.pkl",    "forms",    FORCE_REBUILD)
bm25_examples, ids_examples = build_bm25_index(df_examples, "text", BM25_DIR / "bm25_examples_v2.pkl", "examples", FORCE_REBUILD)

print("\n✅ All BM25 indexes ready.")

# Metadata lookup: chunk_id → row dict (dùng cho BM25-only hits)
legal_meta    = df_legal.set_index("chunk_id").to_dict(orient="index")
forms_meta    = df_forms.set_index("chunk_id").to_dict(orient="index")
examples_meta = df_examples.set_index("chunk_id").to_dict(orient="index")
print("Metadata lookups built.")

Loading chunk files...
  legal    : 237,918 chunks
  forms    : 10 chunks
  examples : 150 chunks

[legal] Loading from cache: dataset/bm25/bm25_legal_v2.pkl
  Loaded 237,918 docs in 11.8s
[forms] Loading from cache: dataset/bm25/bm25_forms_v2.pkl
  Loaded 10 docs in 0.0s
[examples] Loading from cache: dataset/bm25/bm25_examples_v2.pkl
  Loaded 150 docs in 0.0s

✅ All BM25 indexes ready.
Metadata lookups built.


## 4. Chunk Expand — Ghép full điều luật từ parquet

Sau retrieve top-3, với mỗi kết quả legal, lấy **tất cả chunk của cùng `doc_id` + `article`** từ parquet,
sort theo `chunk_index`, ghép lại → LLM nhận full nội dung điều luật thay vì mảnh.

In [7]:
# Build index nhanh để expand: (doc_id, article) → sorted chunks
print("Building expand index...")
t0 = time.time()

# Group by (doc_id, article) → list of (chunk_index, text)
_expand_index: Dict[Tuple[str, str], List[Tuple[int, str]]] = {}
for row in df_legal.itertuples(index=False):
    key = (row.doc_id, row.article)
    if key not in _expand_index:
        _expand_index[key] = []
    _expand_index[key].append((row.chunk_index, row.text))

# Sort theo chunk_index
for key in _expand_index:
    _expand_index[key].sort(key=lambda x: x[0])

print(f"✅ Expand index: {len(_expand_index):,} (doc_id, article) pairs | {time.time()-t0:.1f}s")


def expand_legal_chunk(metadata: Dict, max_chunks: int = MAX_EXPAND_CHUNKS) -> str:
    """
    Lấy full text của điều luật từ expand index.
    Ghép tất cả chunk theo thứ tự chunk_index, tối đa max_chunks.

    Args:
        metadata  : metadata dict từ retrieve result (có doc_id, article)
        max_chunks: giới hạn số chunk tối đa (~400 từ/chunk × max_chunks)

    Returns:
        Full text của điều luật (hoặc text gốc nếu không tìm thấy)
    """
    doc_id  = metadata.get("doc_id", "")
    article = metadata.get("article", "")
    key     = (doc_id, article)

    if key not in _expand_index:
        return metadata.get("text", "")

    chunks = _expand_index[key][:max_chunks]
    texts  = [text for _, text in chunks]

    if len(texts) == 1:
        return texts[0]

    # Ghép: bỏ header lặp lại ở chunk 2+ (đã có article prefix từ chunking)
    # Chunk đầu giữ nguyên, các chunk sau bỏ dòng đầu nếu trùng với article
    merged = [texts[0]]
    for t in texts[1:]:
        lines = t.split("\n")
        # Bỏ dòng đầu nếu là article header (dạng [Điều X. ...])
        if lines and lines[0].startswith("[") and article in lines[0]:
            t = "\n".join(lines[1:]).strip()
        if t:
            merged.append(t)

    return "\n".join(merged)


# Test
test_meta = {"doc_id": df_legal.iloc[0]["doc_id"], "article": df_legal.iloc[0]["article"]}
expanded  = expand_legal_chunk(test_meta)
n_chunks  = len(_expand_index.get((test_meta["doc_id"], test_meta["article"]), []))
print(f"\nTest expand: {n_chunks} chunk(s) → {len(expanded.split())} từ")

Building expand index...
✅ Expand index: 168,946 (doc_id, article) pairs | 1.0s

Test expand: 1 chunk(s) → 54 từ


## 5. Load Embedding Model & ChromaDB

In [8]:
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
t0 = time.time()
embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_EMBED),
)
try:
    EMBED_DIM = embed_model.get_embedding_dimension()
except AttributeError:
    EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"✅ Loaded in {time.time()-t0:.1f}s | dim={EMBED_DIM} | device={DEVICE}")

Loading embedding model: intfloat/multilingual-e5-large-instruct


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Loaded in 15.8s | dim=1024 | device=cuda


In [9]:
print(f"Connecting to ChromaDB: {CHROMA_DIR}")
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

col_legal_all = chroma_client.get_collection("legal_chunks")
col_forms     = chroma_client.get_collection("forms_chunks")
col_examples  = chroma_client.get_collection("examples_chunks")

print(f"✅ legal_chunks    : {col_legal_all.count():,} docs")
print(f"✅ forms_chunks    : {col_forms.count():,} docs")
print(f"✅ examples_chunks : {col_examples.count():,} docs")

Connecting to ChromaDB: dataset/chromadb
✅ legal_chunks    : 237,918 docs
✅ forms_chunks    : 10 docs
✅ examples_chunks : 150 docs


## 6. Load Reranker

In [10]:
print(f"Loading reranker: {RERANK_MODEL_NAME}")
t0 = time.time()
reranker = CrossEncoder(
    RERANK_MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_RERANK),
    max_length=512,
)
print(f"✅ Reranker loaded in {time.time()-t0:.1f}s")

Loading reranker: BAAI/bge-reranker-v2-m3


The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker loaded in 12.6s


## 7. Forms Keyword Pre-filter

In [11]:
FORM_KEYWORD_MAP: List[Tuple[str, List[str]]] = [
    # Form_01 — Nghị quyết cá biệt
    (r"nghi\s*quyet",                                               ["Form_01"]),
    # Form_02 — Quyết định cá biệt | Form_03 — Quyết định ban hành VB kèm theo
    # Route cả hai → reranker chọn đúng dựa trên context
    (r"quyet\s*dinh",                                               ["Form_02", "Form_03"]),
    # Form_04 — Mẫu đa năng (17 loại VB có tên loại chung)
    (r"to\s*trinh|trinh\s*duyet|trinh\s*phe\s*duyet",              ["Form_04"]),
    (r"bao\s*cao",                                                  ["Form_04"]),
    (r"chi\s*thi",                                                  ["Form_04"]),
    (r"thong\s*bao|thong\s*cao",                                    ["Form_04"]),
    (r"huong\s*dan",                                                ["Form_04"]),
    (r"ke\s*hoach|chuong\s*trinh|phuong\s*an|de\s*an|du\s*an",     ["Form_04"]),
    (r"quy\s*che|quy\s*dinh",                                       ["Form_04"]),
    (r"uy\s*quyen|giay\s*uy\s*quyen",                               ["Form_04"]),
    (r"phieu\s*gui|phieu\s*chuyen|phieu\s*bao",                     ["Form_04"]),
    # Form_05 — Công văn
    (r"cong\s*van",                                                 ["Form_05"]),
    # Form_06 — Công điện (thông báo khẩn)
    (r"cong\s*dien|khan\s*cap|thong\s*bao\s*khan",                  ["Form_06"]),
    # Form_07 — Giấy mời họp/hội nghị/hội thảo
    (r"giay\s*moi|thu\s*moi|moi\s*hop|moi\s*hoi\s*nghi",           ["Form_07"]),
    # Form_08 — Giấy giới thiệu (bỏ "cong_tac" — quá chung)
    (r"giay\s*gioi\s*thieu|gioi\s*thieu\s*can\s*bo",                ["Form_08"]),
    # Form_09 — Biên bản họp/hội nghị
    (r"bien\s*ban",                                                 ["Form_09"]),
    # Form_10 — Giấy nghỉ phép
    (r"giay\s*nghi\s*phep|nghi\s*phep|xin\s*nghi",                 ["Form_10"]),
]

_COMPILED_FORM_PATTERNS = [
    (re.compile(pat, re.IGNORECASE), form_ids)
    for pat, form_ids in FORM_KEYWORD_MAP
]


def detect_form_candidates(query: str) -> Optional[List[str]]:
    """
    Normalize query -> bo dau -> match regex -> tra List[form_id] hoac None.
    None = khong detect duoc -> caller dung fallback dense search.
    """
    q_norm = remove_diacritics(normalize_text(query))
    candidates = []
    for pattern, form_ids in _COMPILED_FORM_PATTERNS:
        if pattern.search(q_norm):
            for fid in form_ids:
                if fid not in candidates:
                    candidates.append(fid)
    return candidates if candidates else None


# Test — bao gồm đủ 10 form
test_cases = [
    ("soạn thảo công văn gửi sở tài chính",   ["Form_05"]),
    ("làm tờ trình đề nghị phê duyệt",         ["Form_04"]),
    ("báo cáo tình hình thực hiện",            ["Form_04"]),
    ("quyết định bổ nhiệm cán bộ",             ["Form_02", "Form_03"]),
    ("nghị quyết phiên họp thường kỳ",         ["Form_01"]),
    ("biên bản cuộc họp",                      ["Form_09"]),
    ("giấy mời họp ban giám đốc",              ["Form_07"]),
    ("công điện khẩn thiên tai",               ["Form_06"]),
    ("giấy nghỉ phép năm",                     ["Form_10"]),
    ("soạn thảo văn bản hành chính",           None),
]
print("Keyword detection test:")
for q, expected in test_cases:
    result = detect_form_candidates(q)
    ok = result == expected
    print(f"  {'✅' if ok else '❌'} '{q}'")
    if not ok:
        print(f"       got={result}  expected={expected}")


Keyword detection test:
  ✅ 'soạn thảo công văn gửi sở tài chính'
  ✅ 'làm tờ trình đề nghị phê duyệt'
  ✅ 'báo cáo tình hình thực hiện'
  ✅ 'quyết định bổ nhiệm cán bộ'
  ✅ 'nghị quyết phiên họp thường kỳ'
  ✅ 'biên bản cuộc họp'
  ✅ 'giấy mời họp ban giám đốc'
  ✅ 'công điện khẩn thiên tai'
  ✅ 'giấy nghỉ phép năm'
  ✅ 'soạn thảo văn bản hành chính'


## 8. HybridRetrieverV3 Class

In [12]:
class HybridRetrieverV3:
    """
    Hybrid retrieval v3: BM25 bigram + ChromaDB dense + RRF + Reranker + Dedup.

    Tối ưu so với v2:
      - BM25_TOP_K/DENSE_TOP_K 50→30 → giảm latency
      - RERANK_TOP_N 20→10 → giảm ~50% thời gian rerank
      - MAX_CHUNKS_PER_DOC=1 + chunk expand thay thế dedup lỏng
      - Forms: skip pipeline khi có keyword match
      - Examples: filter theo form candidate từ đầu
    """

    def __init__(
        self,
        bm25_index: BM25Okapi,
        bm25_ids: List[str],
        chroma_col,
        meta_lookup: Dict,
        embed_model: SentenceTransformer,
        reranker: CrossEncoder,
        instruction: str,
        collection_name: str,
        bm25_top_k: int       = BM25_TOP_K,
        dense_top_k: int      = DENSE_TOP_K,
        rrf_k: int            = RRF_K,
        rerank_top_n: int     = RERANK_TOP_N,
        final_top_k: int      = FINAL_TOP_K,
        max_chunks_per_doc: int = MAX_CHUNKS_PER_DOC,
    ):
        self.bm25               = bm25_index
        self.bm25_ids           = bm25_ids
        self.col                = chroma_col
        self.meta               = meta_lookup
        self.embed_model        = embed_model
        self.reranker           = reranker
        self.instruction        = instruction
        self.name               = collection_name
        self.bm25_top_k         = bm25_top_k
        self.dense_top_k        = dense_top_k
        self.rrf_k              = rrf_k
        self.rerank_top_n       = rerank_top_n
        self.final_top_k        = final_top_k
        self.max_chunks_per_doc = max_chunks_per_doc

    def _bm25_search(self, query: str) -> List[Tuple[str, float]]:
        tokens = tokenize_for_bm25(query, use_bigrams=True)
        if not tokens:
            return []
        scores  = self.bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][: self.bm25_top_k]
        return [
            (self.bm25_ids[i], float(scores[i]))
            for i in top_idx if scores[i] > 0
        ]

    def _dense_search(
        self, query: str, where: Optional[Dict] = None
    ) -> List[Tuple[str, float, str, Dict]]:
        q_vec = self.embed_model.encode(
            [self.instruction + query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]
        kwargs = dict(
            query_embeddings=[q_vec.tolist()],
            n_results=self.dense_top_k,
            include=["documents", "metadatas", "distances"],
        )
        if where:
            kwargs["where"] = where
        res = self.col.query(**kwargs)
        return [
            (cid, float(dist), doc, meta or {})
            for cid, dist, doc, meta in zip(
                res["ids"][0], res["distances"][0],
                res["documents"][0], res["metadatas"][0]
            )
        ]

    def _rrf(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float, str, Dict]],
    ) -> List[Tuple[str, float]]:
        scores: Dict[str, float] = {}
        for rank, (cid, _) in enumerate(bm25_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)
        for rank, (cid, _, _, _) in enumerate(dense_results, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (self.rrf_k + rank)
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

    def _rerank(
        self, query: str, candidate_ids: List[str], doc_texts: Dict[str, str]
    ) -> List[Tuple[str, float]]:
        pairs = [(query, doc_texts[cid]) for cid in candidate_ids if cid in doc_texts]
        if not pairs:
            return [(cid, 0.0) for cid in candidate_ids]
        scores = self.reranker.predict(pairs, show_progress_bar=False)
        return sorted(
            zip(candidate_ids[: len(pairs)], scores.tolist()),
            key=lambda x: x[1], reverse=True,
        )

    def _dedup_by_doc(
        self, ranked: List[Tuple[str, float]], chroma_meta: Dict[str, Dict]
    ) -> List[Tuple[str, float]]:
        seen: Dict[str, int] = {}
        result = []
        for cid, score in ranked:
            doc_id = chroma_meta.get(cid, {}).get("doc_id", cid)
            if seen.get(doc_id, 0) < self.max_chunks_per_doc:
                result.append((cid, score))
                seen[doc_id] = seen.get(doc_id, 0) + 1
        return result

    def retrieve(
        self,
        query: str,
        type_filter: Optional[str] = None,
        form_candidates: Optional[List[str]] = None,
        use_reranker: bool = True,
    ) -> List[Dict]:
        t0 = time.time()

        # Where filter cho dense search
        where: Optional[Dict] = None
        if type_filter:
            where = {"type_normalized": type_filter}

        # Form candidates: skip BM25, chỉ dense với where filter
        if form_candidates:
            where = (
                {"form_id": form_candidates[0]}
                if len(form_candidates) == 1
                else {"form_id": {"$in": form_candidates}}
            )
            bm25_res  = []   # skip BM25 hoàn toàn
            dense_res = self._dense_search(query, where=where)
        else:
            bm25_res  = self._bm25_search(query)
            dense_res = self._dense_search(query, where=where)

        # Build text & metadata lookups
        doc_texts:   Dict[str, str]  = {}
        chroma_meta: Dict[str, Dict] = {}
        for cid, _, doc, meta in dense_res:
            doc_texts[cid]   = doc
            chroma_meta[cid] = meta
        for cid, _ in bm25_res:
            if cid not in doc_texts and cid in self.meta:
                doc_texts[cid]   = self.meta[cid].get("text", "")
                chroma_meta[cid] = self.meta[cid]

        bm25_score_map = {cid: s for cid, s in bm25_res}
        dense_dist_map = {cid: d for cid, d, _, _ in dense_res}

        # RRF
        rrf_ranked    = self._rrf(bm25_res, dense_res)
        top_n_ids     = [cid for cid, _ in rrf_ranked[: self.rerank_top_n]]
        rrf_score_map = {cid: s for cid, s in rrf_ranked}

        # Rerank
        if use_reranker and top_n_ids:
            final_ranked = self._rerank(query, top_n_ids, doc_texts)
        else:
            final_ranked = [(cid, rrf_score_map.get(cid, 0.0)) for cid in top_n_ids]

        # Dedup (MAX=1 per doc_id)
        deduped = self._dedup_by_doc(final_ranked, chroma_meta)

        results = []
        for cid, rerank_score in deduped[: self.final_top_k]:
            meta = chroma_meta.get(cid, {})
            results.append({
                "chunk_id"    : cid,
                "text"        : doc_texts.get(cid, ""),
                "metadata"    : meta,
                "bm25_score"  : round(bm25_score_map.get(cid, 0.0), 4),
                "dense_dist"  : round(dense_dist_map.get(cid, 1.0), 4),
                "rrf_score"   : round(rrf_score_map.get(cid, 0.0), 6),
                "rerank_score": round(float(rerank_score), 4),
                "latency_ms"  : round((time.time() - t0) * 1000, 1),
            })
        return results


print("HybridRetrieverV3 class defined.")

HybridRetrieverV3 class defined.


## 9. Khởi tạo Retrievers

In [13]:
retriever_legal = HybridRetrieverV3(
    bm25_index      = bm25_legal,
    bm25_ids        = ids_legal,
    chroma_col      = col_legal_all,
    meta_lookup     = legal_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["legal"],
    collection_name = "legal",
)

retriever_forms = HybridRetrieverV3(
    bm25_index      = bm25_forms,
    bm25_ids        = ids_forms,
    chroma_col      = col_forms,
    meta_lookup     = forms_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["forms"],
    collection_name = "forms",
    final_top_k     = 1,
    max_chunks_per_doc = 1,
)

retriever_examples = HybridRetrieverV3(
    bm25_index      = bm25_examples,
    bm25_ids        = ids_examples,
    chroma_col      = col_examples,
    meta_lookup     = examples_meta,
    embed_model     = embed_model,
    reranker        = reranker,
    instruction     = INSTRUCTIONS["examples"],
    collection_name = "examples",
    final_top_k     = 3,
    max_chunks_per_doc = 1,
)

print("✅ All retrievers v3 ready.")

✅ All retrievers v3 ready.


## 10. Public API

Interface dùng trong RAG generation pipeline.

In [14]:
def retrieve_legal(
    query: str,
    top_k: int = FINAL_TOP_K,
    type_filter: Optional[str] = None,
    use_reranker: bool = True,
    expand: bool = True,
) -> List[Dict]:
    """
    Retrieve dieu khoan phap luat lien quan.
    expand=True: tra full text dieu luat (recommended).
    type_filter: "LUAT" | "NGHI DINH" | "NGHI QUYET" | "PHAP LENH"
    """
    retriever_legal.final_top_k = top_k
    results = retriever_legal.retrieve(
        query, type_filter=type_filter, use_reranker=use_reranker
    )

    if not expand:
        return results

    for r in results:
        doc_id  = r["metadata"].get("doc_id", "")
        article = r["metadata"].get("article", "")
        key     = (doc_id, article)
        n_chunks = len(_expand_index.get(key, []))
        r["text"]             = expand_legal_chunk(r["metadata"])
        r["n_chunks_expanded"] = min(n_chunks, MAX_EXPAND_CHUNKS)

    return results


def retrieve_forms(
    query: str,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Retrieve bieu mau hanh chinh phu hop nhat.
    Logic:
      - Khong detect duoc keyword -> fallback dense search toan bo
      - 1 candidate               -> tra thang (skip reranker)
      - Nhieu candidates          -> fetch tat ca -> reranker chon tot nhat
    Returns: List[Dict] voi 1 form duy nhat.
    """
    form_candidates = detect_form_candidates(query)

    # Khong detect duoc -> dense search toan bo
    if not form_candidates:
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    # 1 candidate -> confident, tra thang
    if len(form_candidates) == 1:
        res = col_forms.get(
            where={"form_id": form_candidates[0]},
            include=["documents", "metadatas"],
        )
        if res["ids"]:
            return [{
                "chunk_id"    : res["ids"][0],
                "text"        : res["documents"][0],
                "metadata"    : res["metadatas"][0],
                "rerank_score": 1.0,
                "source"      : "keyword_match",
            }]
        # form_id khong ton tai trong DB -> fallback
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    # Nhieu candidates (VD: Form_02 + Form_03) -> fetch tat ca -> rerank
    res = col_forms.get(
        where={"form_id": {"$in": form_candidates}},
        include=["documents", "metadatas"],
    )
    if not res["ids"]:
        return retriever_forms.retrieve(query, use_reranker=use_reranker)

    candidates_list = [
        {
            "chunk_id": cid,
            "text"    : doc,
            "metadata": meta,
            "source"  : "keyword_match",
        }
        for cid, doc, meta in zip(res["ids"], res["documents"], res["metadatas"])
    ]

    if use_reranker and len(candidates_list) > 1:
        pairs = [(query, c["text"]) for c in candidates_list]
        scores = reranker.predict(pairs)
        for c, s in zip(candidates_list, scores):
            c["rerank_score"] = float(s)
        candidates_list.sort(key=lambda x: x["rerank_score"], reverse=True)
    else:
        for c in candidates_list:
            c["rerank_score"] = 1.0

    return [candidates_list[0]]


def retrieve_examples(
    query: str,
    top_k: int = 3,
    form_id: Optional[str] = None,
    use_reranker: bool = True,
) -> List[Dict]:
    """
    Retrieve vi du van ban (few-shot examples).
    Uu tien: form_id truyen thang > keyword detect > fallback dense.
    """
    retriever_examples.final_top_k = top_k

    # Goi detect_form_candidates 1 lan duy nhat (tranh double-call)
    if not form_id:
        _candidates = detect_form_candidates(query)
        form_id = _candidates[0] if _candidates else None

    return retriever_examples.retrieve(
        query,
        form_candidates=[form_id] if form_id else None,
        use_reranker=use_reranker,
    )


def retrieve_all(
    query: str,
    legal_top_k: int = FINAL_TOP_K,
    examples_top_k: int = 3,
    legal_type_filter: Optional[str] = None,
    expand_legal: bool = True,
) -> Dict[str, List[Dict]]:
    """
    Entry point chinh cho RAG generation pipeline.
    Returns: {"legal": [...], "form": [...], "examples": [...]}
    """
    form_results = retrieve_forms(query)

    # Lay form_id tu ket qua forms de pass vao examples (tranh double detect)
    detected_form_id = (
        form_results[0]["metadata"].get("form_id")
        if form_results else None
    )

    return {
        "legal"   : retrieve_legal(
            query, top_k=legal_top_k,
            type_filter=legal_type_filter,
            expand=expand_legal,
        ),
        "form"    : form_results,
        "examples": retrieve_examples(
            query, top_k=examples_top_k,
            form_id=detected_form_id,
        ),
    }


print("✅ Public API defined.")


✅ Public API defined.


## 11. Test & Benchmark

In [15]:
def print_legal_results(results: List[Dict], query: str):
    print(f"\n{'='*70}")
    print(f"[LEGAL] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta     = r["metadata"]
        n_chunks = r.get("n_chunks_expanded", 1)
        n_words  = len(r["text"].split())
        print(f"\n  [{i}] {meta.get('source_doc_no','?')} | {meta.get('type_normalized','')}")
        print(f"      📌 {meta.get('article','')}")
        print(f"      🎯 rerank={r.get('rerank_score',0):.4f} | bm25={r.get('bm25_score',0):.2f} | "
              f"dense_dist={r.get('dense_dist',1):.4f}")
        print(f"      📦 {n_chunks} chunk(s) expanded → {n_words} từ")
        print(f"      📝 {r['text'][:200].replace(chr(10),' ')}...")
    if results:
        print(f"\n  ⏱  Latency: {results[0].get('latency_ms',0)}ms")


def print_forms_results(results: List[Dict], query: str):
    print(f"\n{'='*70}")
    print(f"[FORMS] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        src  = r.get("source", "pipeline")
        print(f"  [{i}] {meta.get('form_id','?')} | {meta.get('form_type','')} | "
              f"rerank={r.get('rerank_score',0):.4f} | source={src}")
        print(f"       {meta.get('purpose','')}")


def print_examples_results(results: List[Dict], query: str):
    print(f"\n{'='*70}")
    print(f"[EXAMPLES] Query: {query}")
    print(f"{'='*70}")
    for i, r in enumerate(results, 1):
        meta = r["metadata"]
        print(f"  [{i}] {meta.get('form_id','?')} | rerank={r.get('rerank_score',0):.4f}")
        print(f"       {meta.get('scenario','')[:120]}")


print("Display helpers defined.")

Display helpers defined.


In [16]:
# ── Test legal + expand ──────────────────────────────────────
legal_queries = [
    "Quy định về hòa giải ở cơ sở",
    "trách nhiệm của Bộ trưởng trong ban hành văn bản",
    "quy trình bổ nhiệm cán bộ công chức",
    "điều kiện thành lập doanh nghiệp",
    "bảo hiểm xã hội bắt buộc cho người lao động",
]

for query in legal_queries:
    results = retrieve_legal(query, expand=True)
    print_legal_results(results, query)
    print("-" * 70)


[LEGAL] Query: Quy định về hòa giải ở cơ sở

  [1] 160/1999/NĐ-CP | NGHỊ ĐỊNH
      📌 Điều 2. Hoà giải ở cơ sở
      🎯 rerank=0.9974 | bm25=0.00 | dense_dist=0.0656
      📦 1 chunk(s) expanded → 194 từ
      📝 Điều 2. Hoà giải ở cơ sở 1. Hoà giải ở cơ sở là việc hướng dẫn, giúp đỡ, thuyết phục các bên tranh chấp đạt được thoả thuận, tự nguyện giải quyết với nhau những việc vi phạm pháp luật và tranh chấp nh...

  [2] 35/2013/QH13 | LUẬT
      📌 Điều 28. Trách nhiệm quản lý nhà nước về hòa giải ở cơ sở
      🎯 rerank=0.9965 | bm25=84.28 | dense_dist=1.0000
      📦 1 chunk(s) expanded → 233 từ
      📝 Điều 28. Trách nhiệm quản lý nhà nước về hòa giải ở cơ sở 1. Chính phủ thống nhất quản lý nhà nước về hòa giải ở cơ sở. 2. Bộ Tư pháp chịu trách nhiệm trước Chính phủ thực hiện quản lý nhà nước về hòa...

  [3] 35/2013/QH13 | LUẬT
      📌 Điều 4. Nguyên tắc tổ chức, hoạt động hòa giải ở cơ sở
      🎯 rerank=0.9955 | bm25=0.00 | dense_dist=0.0591
      📦 1 chunk(s) expanded → 239 từ
      📝

In [17]:
# ── Test forms & examples ────────────────────────────────────
form_queries = [
    "soạn thảo công văn gửi sở tài chính",
    "làm tờ trình đề nghị phê duyệt dự án",
    "báo cáo tình hình thực hiện nhiệm vụ",
    "viết giấy mời họp ban lãnh đạo",
    "quyết định bổ nhiệm trưởng phòng",
    "giấy giới thiệu cán bộ đi công tác",
]

for query in form_queries:
    res_forms    = retrieve_forms(query)
    res_examples = retrieve_examples(query)
    print_forms_results(res_forms, query)
    print_examples_results(res_examples, query)


[FORMS] Query: soạn thảo công văn gửi sở tài chính
  [1] Form_05 | Công văn | rerank=1.0000 | source=keyword_match
       Giao dịch, trao đổi công việc hành chính

[EXAMPLES] Query: soạn thảo công văn gửi sở tài chính
  [1] Form_05 | rerank=0.1039
       Cục Thuế tỉnh Nghệ An đề nghị hộ kinh doanh triển khai sử dụng hóa đơn điện tử khởi tạo từ máy tính tiền trong năm 2025.
  [2] Form_05 | rerank=0.0385
       Sở Tài nguyên và Môi trường Bắc Ninh yêu cầu các khu công nghiệp tăng cường quan trắc nước thải tự động năm 2023.
  [3] Form_05 | rerank=0.0255
       Trường Đại học Cần Thơ gửi công văn đề nghị các khoa cập nhật minh chứng kiểm định chất lượng chương trình đào tạo năm 2

[FORMS] Query: làm tờ trình đề nghị phê duyệt dự án
  [1] Form_04 | Chỉ thị, Quy chế, Quy định, Thông cáo, Thông báo, Hướng dẫn, Chương trình, Kế hoạch, Phương án, Đề án, Dự án, Báo cáo, Tờ trình, Giấy ủy quyền, Phiếu gửi, Phiếu chuyển, Phiếu báo | rerank=1.0000 | source=keyword_match
       Văn bản hành chính c

In [18]:
# ── Benchmark BM25 + dedup + latency ────────────────────────
benchmark_queries = [
    "quy dinh ve hoa giai o co so",
    "dieu kien de duoc cap giay phep kinh doanh",
    "trach nhiem cua Bo truong trong ban hanh van ban",
    "xu phat vi pham hanh chinh giao thong duong bo",
    "quyen va nghia vu nguoi su dung dat",
    "thu tuc dang ky kinh doanh ho ca the",
    "chinh sach ho tro nguoi co cong voi cach mang",
    "quy trinh kiem tra chat luong hang hoa xuat nhap khau",
]

print(f"{'Query':<50} {'BM25':>8} {'Rerank':>8} {'Expanded':>10} {'Latency':>10}")
print("-" * 92)

for q in benchmark_queries:
    results = retrieve_legal(q, expand=True)
    if not results:
        print(f"{q[:49]:<50} N/A")
        continue
    r        = results[0]
    bm25_flag = "✅" if r["bm25_score"] > 0 else "❌"
    n_words   = len(r["text"].split())
    print(
        f"{q[:49]:<50} "
        f"{bm25_flag} {r['bm25_score']:>5.1f} "
        f"{r['rerank_score']:>8.4f} "
        f"{n_words:>8}từ "
        f"{r['latency_ms']:>8.0f}ms"
    )

Query                                                  BM25   Rerank   Expanded    Latency
--------------------------------------------------------------------------------------------
quy dinh ve hoa giai o co so                       ❌   0.0   0.0058      256từ     3761ms
dieu kien de duoc cap giay phep kinh doanh         ✅  59.0   0.4073      223từ     3969ms
trach nhiem cua Bo truong trong ban hanh van ban   ✅  65.1   0.9608       78từ     4128ms
xu phat vi pham hanh chinh giao thong duong bo     ✅  69.9   0.5988      199từ     1970ms
quyen va nghia vu nguoi su dung dat                ✅  58.6   0.5757       59từ     3349ms
thu tuc dang ky kinh doanh ho ca the               ✅  63.1   0.9565      550từ     3936ms
chinh sach ho tro nguoi co cong voi cach mang      ✅  92.1   0.7855       60từ     4089ms
quy trinh kiem tra chat luong hang hoa xuat nhap   ✅  76.2   0.7534      372từ     4347ms


In [19]:
# ── Demo retrieve_all ────────────────────────────────────────
demo_query = "soạn thảo quyết định bổ nhiệm cán bộ"
print(f"Demo retrieve_all(): '{demo_query}'\n")

t0  = time.time()
all_results = retrieve_all(demo_query)
elapsed = (time.time() - t0) * 1000

print(f"  Legal    : {len(all_results['legal'])} điều luật")
print(f"  Form     : {len(all_results['form'])} form")
print(f"  Examples : {len(all_results['examples'])} ví dụ")
print(f"  Total latency: {elapsed:.0f}ms")

if all_results["legal"]:
    r = all_results["legal"][0]
    print(f"\nTop legal : {r['metadata'].get('source_doc_no','?')} | {r['metadata'].get('article','')}")
    print(f"  rerank={r['rerank_score']:.4f} | {r.get('n_chunks_expanded',1)} chunk(s) | {len(r['text'].split())} từ")

if all_results["form"]:
    r = all_results["form"][0]
    print(f"Top form  : {r['metadata'].get('form_id','?')} | {r['metadata'].get('form_type','')} | source={r.get('source','pipeline')}")

if all_results["examples"]:
    r = all_results["examples"][0]
    print(f"Top example: {r['metadata'].get('form_id','?')} | rerank={r.get('rerank_score',0):.4f}")
    print(f"  {r['metadata'].get('scenario','')[:100]}")

Demo retrieve_all(): 'soạn thảo quyết định bổ nhiệm cán bộ'

  Legal    : 3 điều luật
  Form     : 1 form
  Examples : 3 ví dụ
  Total latency: 4683ms

Top legal : 43/2009/QH12 | Điều 29. Thẩm quyền bổ nhiệm cán bộ Ban chỉ huy quân sự bộ, ngành trung ương, cán bộ chỉ huy quân sự cơ sở và cán bộ dân quân tự vệ
  rerank=0.8256 | 1 chunk(s) | 358 từ
Top form  : Form_02 | Quyết định | source=keyword_match
Top example: Form_02 | rerank=0.8493
  Bộ Tài chính bổ nhiệm Vụ trưởng Vụ Chính sách Thuế.


---
## Summary

| Component | v2 | v3 |
|---|---|---|
| **BM25_TOP_K** | 50 | **30** |
| **DENSE_TOP_K** | 50 | **30** |
| **RERANK_TOP_N** | 20 | **10** (~2x nhanh hơn) |
| **FINAL_TOP_K** | 5 | **3** |
| **MAX_CHUNKS_PER_DOC** | 2 | **1** (thay bằng expand) |
| **Chunk expand** | ❌ | ✅ ghép full điều luật từ parquet |
| **Forms pipeline** | BM25+Dense+Rerank | **Direct lookup** khi keyword match |
| **Examples filter** | fallback | **Filter theo form candidate** |

**Public API:**
```python
retrieve_all(query)           # entry point chính
retrieve_legal(query, expand=True)
retrieve_forms(query)
retrieve_examples(query, form_id=None)
```